# Análisis del Motor de Búsqueda Semántica

Este notebook visualiza los resultados del motor de búsqueda semántica creado para el laboratorio.

In [ ]:
# Importar librerías necesarias
import os
import chromadb
from utils import crear_embedding
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo de gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar variables de entorno
load_dotenv()

# Inicializar cliente de ChromaDB
chroma_client = chromadb.PersistentClient(path="./db")
collection = chroma_client.get_collection(name="articulos")
print(f"Colección cargada con {collection.count()} documentos")

In [ ]:
# Definir las queries de prueba
test_queries = [
    "como hacer una API en Python",
    "diferencias entre frameworks de frontend",
    "como funciona la autenticacion en aplicaciones web",
    "herramientas para trabajar con modelos de lenguaje"
]

# Función para buscar y obtener resultados
def buscar_con_detalles(query, n_resultados=3):
    embedding_query = crear_embedding(query)
    resultados = collection.query(
        query_embeddings=[embedding_query],
        n_results=n_resultados
    )
    
    documentos = resultados["documents"][0]
    metadatos = resultados["metadatas"][0]
    distancias = resultados["distances"][0]
    
    resultados_formateados = []
    for i in range(len(documentos)):
        similitud = 1 - distancias[i]
        similitud = max(0, min(1, similitud))  # Asegurar que esté entre 0 y 1
        resultados_formateados.append({
            'query': query,
            'resultado': metadatos[i]['titulo'],
            'score': similitud,
            'contenido': documentos[i]
        })
    return resultados_formateados

# Ejecutar búsquedas
todos_los_resultados = []
for query in test_queries:
    resultados = buscar_con_detalles(query)
    todos_los_resultados.extend(resultados)

# Convertir a DataFrame
df_resultados = pd.DataFrame(todos_los_resultados)
df_resultados